# Import

In [4]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:157: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


# Download data

In [4]:
!aws s3 cp --no-sign-request s3://genome-scale-tcell-perturb-seq/marson2025_data/D1_Rest.assigned_guide.h5ad /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/zhu_2025_D1_rest_cl.h5ad
# !mv /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/D1_Rest.assigned_guide.h5ad /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/zhu_2025_pseudobulk.h5ad

download: s3://genome-scale-tcell-perturb-seq/marson2025_data/D1_Rest.assigned_guide.h5ad to Perturbseq/non_curated/h5ad/zhu_2025_D1_rest_cl.h5ad


# Initialise the dataset object

In [3]:
noncurated_path = '/content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/zhu_2025_D1_rest_cl.h5ad'
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/zhu_2025_D1_rest_cl.h5ad


In [4]:
cur_data.adata.obs

,lane_id,n_genes_by_counts,total_counts,pct_counts_mt,top_guide_UMI_counts,guide_id,perturbed_gene_name,perturbed_gene_id,guide_type,PuroR,guide_group,low_quality
AAACAAGCAAACCGGTAACGGGAA-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,3945,8874.0,0.439486,169.0,multi_sgRNA,NaN,NaN,targeting,0.587712,multi sgRNA,False
AAACAAGCAAACCGGTATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,1953,2969.0,0.269451,NaN,NaN,NaN,NaN,NaN,2.100727,no sgRNA,False
AAACAAGCAAAGGGATAGTAGGCT-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,2989,5807.0,0.654383,75.0,multi_sgRNA,NaN,NaN,targeting,2.826730,multi sgRNA,False
AAACAAGCAAATACCGATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,4576,13363.0,1.017735,22.0,NTC-461,NTC,NTC,non-targeting,1.431987,targeting single sgRNA,False
AAACAAGCAAATCACGATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,1890,3016.0,0.397878,9.0,PLA2G10-2,PLA2G10,ENSG00000069764,targeting,1.209989,targeting single sgRNA,False
...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGAGTTGTGACTAACGGGAA-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,2725,5182.0,0.135083,NaN,NaN,NaN,NaN,NaN,1.451924,no sgRNA,False
TTTGTGAGTTGTGACTATGTTGAC-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,4145,10171.0,0.304788,NaN,NaN,NaN,NaN,NaN,1.466352,no sgRNA,False
TTTGTGAGTTTAACCAAGTAGGCT-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,3504,9302.0,0.225758,NaN,NaN,NaN,NaN,NaN,0.000000,no sgRNA,False
TTTGTGAGTTTCGCCTACTTTAGG-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,2624,4919.0,0.203293,25.0,KIF18A-2,KIF18A,ENSG00000121621,targeting,1.492056,targeting single sgRNA,False


In [5]:
cur_data.adata.var

,gene_ids,feature_types,genome,gene_name,mt
ENSG00000000003,ENSG00000000003,Gene Expression,GRCh38,TSPAN6,False
ENSG00000000005,ENSG00000000005,Gene Expression,GRCh38,TNMD,False
ENSG00000000419,ENSG00000000419,Gene Expression,GRCh38,DPM1,False
ENSG00000000457,ENSG00000000457,Gene Expression,GRCh38,SCYL3,False
ENSG00000000460,ENSG00000000460,Gene Expression,GRCh38,C1orf112,False
...,...,...,...,...,...
ENSG00000291122,ENSG00000291122,Gene Expression,GRCh38,CASTOR3P,False
ENSG00000291135,ENSG00000291135,Gene Expression,GRCh38,FCGR1BP,False
ENSG00000291145,ENSG00000291145,Gene Expression,GRCh38,PPP5D1P,False
ENSG00000291237,ENSG00000291237,Gene Expression,GRCh38,SOD2,False


# OBS slot curation

### Since for multi-guide perturbations the identities of said guides are unknown, we are filtering out these cells.

In [6]:
print(f"Number of cells before filtering: {cur_data.adata.n_obs}")
cur_data.adata = cur_data.adata[cur_data.adata.obs['guide_id'] != 'multi_sgRNA']
print(f"Number of cells after filtering: {cur_data.adata.n_obs}")

Number of cells before filtering: 3074496
Number of cells after filtering: 2548393


### Add `index` as `perturbation_name`

In [ ]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]
cur_data.adata.obs['perturbation_name'] = cur_data.adata.obs['cell_barcode'] +'_'+cur_data.adata.obs['lane_id'].astype(str)

/tmp/ipython-input-2231587657.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]


### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

### Add guide RNA information

In [27]:
# download the guide RNA spreadsheet
download_file(
    url="https://raw.githubusercontent.com/emdann/GWT_perturbseq_analysis_2025/refs/heads/master/metadata/suppl_tables/sgrna_library_metadata.suppl_table.csv",
    dest_path="/content/PerturbationCatalogue/data_exploration/Perturbseq/supplementary/zhu_2025_guide_info.csv"
)
# read in the guide RNA info csv
guide_info_df = pd.read_csv("/content/PerturbationCatalogue/data_exploration/Perturbseq/supplementary/zhu_2025_guide_info.csv")

guide_info_df = guide_info_df[['sgRNA', 'seq']]

guide_info_df['sgRNA'] = guide_info_df['sgRNA'].str.replace('1-Jun','JUN-1').str.replace('2-Jun','JUN-2')

guide_info_df = guide_info_df.rename(columns={'seq': 'guide_sequence', 'sgRNA':'guide_id'})

guide_info_df


Downloaded https://raw.githubusercontent.com/emdann/GWT_perturbseq_analysis_2025/refs/heads/master/metadata/suppl_tables/sgrna_library_metadata.suppl_table.csv to /content/PerturbationCatalogue/data_exploration/Perturbseq/supplementary/zhu_2025_guide_info.csv


,guide_id,guide_sequence
0,ARMC5-1,CTGCCTCGCGCAGCTCGCGG
1,DDB1-1,GGAGTTCGCTGCGCGCTGTT
2,FNDC10-1,TGCCCGCTCCCCGCGATCCC
3,RORC-1,TCTGTGGGGCCCTGTCCATG
4,SLC35F6-1,GAACAGCTGGTACTTGGTCC
...,...,...
26499,NTC-988,TTTCAGGCTACGGGCGCGGG
26500,NTC-989,TTTCCCATGATCATTTAGTG
26501,NTC-990,TTTCGCCCAAGAGGCTTGGG
26502,NTC-991,TTTCGTGCCGATGTAACACA


In [28]:
print("Number of overlapping guides with cur_data:")
cur_data.adata.obs['guide_id'].isin(guide_info_df['guide_id'].to_list()).value_counts()

Number of overlapping guides with cur_data:


,count
guide_id,
True,1754014
False,794379


In [ ]:
# merge cur_data.adata.obs with guide_info_df
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='guide_id', how='left')
# check that there are no missing guide sequences
print(f"Number of missing guide sequences: {cur_data.adata.obs['guide_sequence'].isna().sum()}")


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
cur_data.adata.obs

,lane_id,n_genes_by_counts,total_counts,pct_counts_mt,top_guide_UMI_counts,guide_id,perturbed_gene_name,perturbed_gene_id,guide_type,PuroR,guide_group,low_quality,perturbation_name,guide_sequence
0,CD4i_R1L08,3945,8874.0,0.439486,169.0,multi_sgRNA,NaN,NaN,targeting,0.587712,multi sgRNA,False,AAACAAGCAAACCGGTAACGGGAA-1_CD4i_R1L08_CD4i_R1_...,NaN
1,CD4i_R1L08,1953,2969.0,0.269451,NaN,NaN,NaN,NaN,NaN,2.100727,no sgRNA,False,AAACAAGCAAACCGGTATGTTGAC-1_CD4i_R1L08_CD4i_R1_...,NaN
2,CD4i_R1L08,2989,5807.0,0.654383,75.0,multi_sgRNA,NaN,NaN,targeting,2.826730,multi sgRNA,False,AAACAAGCAAAGGGATAGTAGGCT-1_CD4i_R1L08_CD4i_R1_...,NaN
3,CD4i_R1L08,4576,13363.0,1.017735,22.0,NTC-461,NTC,NTC,non-targeting,1.431987,targeting single sgRNA,False,AAACAAGCAAATACCGATGTTGAC-1_CD4i_R1L08_CD4i_R1_...,TGGGAATTCCTCGGCCGATT
4,CD4i_R1L08,1890,3016.0,0.397878,9.0,PLA2G10-2,PLA2G10,ENSG00000069764,targeting,1.209989,targeting single sgRNA,False,AAACAAGCAAATCACGATGTTGAC-1_CD4i_R1L08_CD4i_R1_...,GGGTGCTGCAGGGCCACCGG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3074491,CD4i_R1L21,2725,5182.0,0.135083,NaN,NaN,NaN,NaN,NaN,1.451924,no sgRNA,False,TTTGTGAGTTGTGACTAACGGGAA-1_CD4i_R1L21_CD4i_R1_...,NaN
3074492,CD4i_R1L21,4145,10171.0,0.304788,NaN,NaN,NaN,NaN,NaN,1.466352,no sgRNA,False,TTTGTGAGTTGTGACTATGTTGAC-1_CD4i_R1L21_CD4i_R1_...,NaN
3074493,CD4i_R1L21,3504,9302.0,0.225758,NaN,NaN,NaN,NaN,NaN,0.000000,no sgRNA,False,TTTGTGAGTTTAACCAAGTAGGCT-1_CD4i_R1L21_CD4i_R1_...,NaN
3074494,CD4i_R1L21,2624,4919.0,0.203293,25.0,KIF18A-2,KIF18A,ENSG00000121621,targeting,1.492056,targeting single sgRNA,False,TTTGTGAGTTTCGCCTACTTTAGG-1_CD4i_R1L21_CD4i_R1_...,AGGCGGACATTAAAGTGAAG


In [ ]:
# change perturbed_gene_id from categorical to string
cur_data.adata.obs['perturbed_gene_id'] = cur_data.adata.obs['perturbed_gene_id'].astype(str)
cur_data.adata.obs['perturbed_gene_name'] = cur_data.adata.obs['perturbed_gene_name'].astype(str)

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isin(['NTC']), 'perturbed_gene_id'] = 'control_nontargeting'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin(['NTC']), 'perturbed_gene_name'] = 'control_nontargeting'

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isna(), 'perturbed_gene_id'] = 'control_casonly'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isna(), 'perturbed_gene_name'] = 'control_casonly'

### Standardise perturbation targets

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='perturbed_gene_id',
    input_column_type='ensembl_gene_id',
    multiple_entries=False,
    # remove_version=True,
    # version_sep='.'
)

Missing Ensembl IDs: ['ENSG00000277796', 'SGK494', 'SEPT6', 'SMIM11B', 'AKAP2', 'ENSG00000255823', 'ENSG00000256618', 'OCLM', 'ENSG00000232196', 'ENSG00000182230', 'ENSG00000148362', 'U2AF1L5', 'ENSG00000203812']; attempting to fetch latest IDs...
Fetched latest Ensembl IDs: {'ENSG00000277796': 'ENSG00000293545', 'ENSG00000255823': nan, 'ENSG00000256618': nan, 'ENSG00000232196': nan, 'ENSG00000182230': nan, 'ENSG00000148362': 'ENSG00000310560', 'ENSG00000203812': 'ENSG00000288825'}
--------------------------------------------------
Successfully mapped 12731 out of 12731 Ensembl IDs.
--------------------------------------------------


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
cur_data.adata.obs

,log10_n_cells,developmental_stage_label,perturbed_gene_id,keep_for_DE,keep_total_counts,10xrun_id,perturbed_gene_name,perturbation_name,guide_id,sex_label,biological_replicate,n_cells,culture_condition,keep_test_genes,total_counts,keep_effective_guides,donor_id,guide_sequence,guide_type,keep_min_cells,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,,,,,,,,,,,,,,,,
0,1.414973,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-1,A1BG-1,female,D1_CE0008162,26.0,Stim48hr,True,326500.0,True,CE0008162,GGACGGCATCTCGGCCCGCC,targeting,True,ENSG00000121410,A1BG,protein_coding,chr19:58345178-58353492;-1,19,0
1,2.204120,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-2,A1BG-2,female,D1_CE0008162,160.0,Stim48hr,True,2313880.0,True,CE0008162,GGGTCCCTCGCAGCGCAGGA,targeting,True,ENSG00000121410,A1BG,protein_coding,chr19:58345178-58353492;-1,19,1
2,0.000000,adult,ENSG00000175899,False,True,CD4i_R2,A2M,CD4i_R2_D1_Stim48hr_A2M-1,A2M-1,female,D1_CE0008162,1.0,Stim48hr,True,23782.0,True,CE0008162,CAGATGGATTGTAGGGAGTA,targeting,False,ENSG00000175899,A2M,protein_coding,chr12:9067664-9116229;-1,12,2
3,1.342423,adult,ENSG00000175899,True,True,CD4i_R2,A2M,CD4i_R2_D1_Stim48hr_A2M-2,A2M-2,female,D1_CE0008162,22.0,Stim48hr,True,330204.0,True,CE0008162,CCAGATGGATTGTAGGGAGT,targeting,True,ENSG00000175899,A2M,protein_coding,chr12:9067664-9116229;-1,12,3
4,1.977724,adult,ENSG00000094914,True,True,CD4i_R2,AAAS,CD4i_R2_D1_Stim48hr_AAAS-1,AAAS-1,female,D1_CE0008162,95.0,Stim48hr,True,1389867.0,True,CE0008162,GGGCTAGATTCGTATGCGGA,targeting,True,ENSG00000094914,AAAS,protein_coding,chr12:53307456-53324864;-1,12,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278679,0.000000,adult,ENSG00000159840,False,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-1,ZYX-1,male,D4_CE0006864,1.0,Rest,True,11009.0,True,CE0006864,GGAGGCGGCCACCCGAGACG,targeting,False,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278679
278680,1.591065,adult,ENSG00000159840,True,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-2,ZYX-2,male,D4_CE0006864,39.0,Rest,True,318693.0,True,CE0006864,GGACCGGGACGCAGAGTCTG,targeting,True,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278680
278681,1.491362,adult,ENSG00000159840,True,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-3,ZYX-3,male,D4_CE0006864,31.0,Rest,True,247297.0,True,CE0006864,GGCCGGAGCCGAGACCGAAA,targeting,True,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278681


# Manually replace some genes

In [ ]:
# define genes to replace in the cur_data.adata.obs
genes_to_replace = cur_data.adata.obs[cur_data.adata.obs['perturbed_target_symbol'].isna()]['perturbed_gene_name'].drop_duplicates().to_list()
# define columns to replace
cols_to_replace = ['perturbed_target_ensg',	'perturbed_target_symbol',	'perturbed_target_biotype',	'perturbed_target_coord',	'perturbed_target_chromosome']
# define mapping dict for renaming columns in gene_ont for replacement
gene_ont_col_mapping = {
    'ensembl_gene_id':'perturbed_target_ensg',
    'gene_symbol':'perturbed_target_symbol',
    'biotype':'perturbed_target_biotype',
    'gene_coord':'perturbed_target_coord',
    'chromosome_name':'perturbed_target_chromosome'
}
# loop through genes and fill in the missing cols
for replacement_gene in genes_to_replace:
    if replacement_gene in cur_data.gene_ont['synonym'].values:
        # subset gene_ont and get values for replacement
        replacement = (
            cur_data.gene_ont.rename(columns=gene_ont_col_mapping)
            .loc[cur_data.gene_ont['synonym'].isin([replacement_gene]), cols_to_replace]
        )[cols_to_replace].values[0]
        # replace values in cur_data.adata.obs with the values from gene_ont
        cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin([replacement_gene]), cols_to_replace] = replacement
        print(f"Replaced {replacement_gene} with {replacement}")
    else:
        print(f"No replacement found for {replacement_gene}")


No replacement found for AKAP2
No replacement found for CCL3L1
No replacement found for FAM153B
No replacement found for MTRNR2L1
No replacement found for MTRNR2L4
No replacement found for MTRNR2L8
No replacement found for OCLM
Replaced SEPT6 with ['ENSG00000125354' 'SEPTIN6' 'protein_coding'
 'chrX:119615724-119693949;-1' 'X']
Replaced SGK494 with ['ENSG00000167524' 'RSKR' 'protein_coding' 'chr17:28455752-28614197;-1'
 '17']
No replacement found for SMIM11B
No replacement found for U2AF1L5


In [ ]:
# replace non-mapped perturbed_target_symbol with original gene symbols
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_target_symbol'] = cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_gene_name'].values

In [ ]:
# replace entries in perturbed_target_ensg that are not ENSEMBL ids with NA
cur_data.adata.obs.loc[(~cur_data.adata.obs['perturbed_target_ensg'].str.startswith('ENSG')) & (cur_data.adata.obs['perturbed_target_ensg'] != 'control_nontargeting'), 'perturbed_target_ensg'] = pd.NA

In [ ]:
cur_data.adata.obs

,log10_n_cells,developmental_stage_label,perturbed_gene_id,keep_for_DE,keep_total_counts,10xrun_id,perturbed_gene_name,perturbation_name,guide_id,sex_label,biological_replicate,n_cells,culture_condition,keep_test_genes,total_counts,keep_effective_guides,donor_id,guide_sequence,guide_type,keep_min_cells,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index
index,,,,,,,,,,,,,,,,,,,,,,,,,,
0,1.414973,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-1,A1BG-1,female,D1_CE0008162,26.0,Stim48hr,True,326500.0,True,CE0008162,GGACGGCATCTCGGCCCGCC,targeting,True,ENSG00000121410,A1BG,protein_coding,chr19:58345178-58353492;-1,19,0
1,2.204120,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-2,A1BG-2,female,D1_CE0008162,160.0,Stim48hr,True,2313880.0,True,CE0008162,GGGTCCCTCGCAGCGCAGGA,targeting,True,ENSG00000121410,A1BG,protein_coding,chr19:58345178-58353492;-1,19,1
2,0.000000,adult,ENSG00000175899,False,True,CD4i_R2,A2M,CD4i_R2_D1_Stim48hr_A2M-1,A2M-1,female,D1_CE0008162,1.0,Stim48hr,True,23782.0,True,CE0008162,CAGATGGATTGTAGGGAGTA,targeting,False,ENSG00000175899,A2M,protein_coding,chr12:9067664-9116229;-1,12,2
3,1.342423,adult,ENSG00000175899,True,True,CD4i_R2,A2M,CD4i_R2_D1_Stim48hr_A2M-2,A2M-2,female,D1_CE0008162,22.0,Stim48hr,True,330204.0,True,CE0008162,CCAGATGGATTGTAGGGAGT,targeting,True,ENSG00000175899,A2M,protein_coding,chr12:9067664-9116229;-1,12,3
4,1.977724,adult,ENSG00000094914,True,True,CD4i_R2,AAAS,CD4i_R2_D1_Stim48hr_AAAS-1,AAAS-1,female,D1_CE0008162,95.0,Stim48hr,True,1389867.0,True,CE0008162,GGGCTAGATTCGTATGCGGA,targeting,True,ENSG00000094914,AAAS,protein_coding,chr12:53307456-53324864;-1,12,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278679,0.000000,adult,ENSG00000159840,False,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-1,ZYX-1,male,D4_CE0006864,1.0,Rest,True,11009.0,True,CE0006864,GGAGGCGGCCACCCGAGACG,targeting,False,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278679
278680,1.591065,adult,ENSG00000159840,True,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-2,ZYX-2,male,D4_CE0006864,39.0,Rest,True,318693.0,True,CE0006864,GGACCGGGACGCAGAGTCTG,targeting,True,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278680
278681,1.491362,adult,ENSG00000159840,True,True,CD4i_R2,ZYX,CD4i_R2_D4_Rest_ZYX-3,ZYX-3,male,D4_CE0006864,31.0,Rest,True,247297.0,True,CE0006864,GGCCGGAGCCGAGACCGAAA,targeting,True,ENSG00000159840,ZYX,protein_coding,chr7:143381295-143391111;1,7,278681


### Add `perturbed_target_number` column

In [ ]:
# cur_data.count_entries(
#     slot='obs',
#     input_column='perturbed_target_ensg',
#     count_column_name='perturbed_target_number',
#     sep='|'
# )
cur_data.adata.obs['perturbed_target_number'] = 1

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


In [ ]:
cur_data.show_obs(['perturbation_name', 'perturbed_target_chromosome_encoding'])

Observation data:
DataFrame shape: (278684, 2)
--------------------------------------------------
                 perturbation_name  perturbed_target_chromosome_encoding
index                                                                   
0       CD4i_R2_D1_Stim48hr_A1BG-1                                    19
1       CD4i_R2_D1_Stim48hr_A1BG-2                                    19
2        CD4i_R2_D1_Stim48hr_A2M-1                                    12
3        CD4i_R2_D1_Stim48hr_A2M-2                                    12
4       CD4i_R2_D1_Stim48hr_AAAS-1                                    12
...                            ...                                   ...
278679       CD4i_R2_D4_Rest_ZYX-1                                     7
278680       CD4i_R2_D4_Rest_ZYX-2                                     7
278681       CD4i_R2_D4_Rest_ZYX-3                                     7
278682     CD4i_R2_D4_Rest_ZZEF1-1                                    17
278683     CD4i_R2_D4_Rest

In [ ]:
cur_data.adata.obs.columns

Index(['log10_n_cells', 'developmental_stage_label', 'perturbed_gene_id',
       'keep_for_DE', 'keep_total_counts', '10xrun_id', 'perturbed_gene_name',
       'perturbation_name', 'guide_id', 'sex_label', 'biological_replicate',
       'n_cells', 'culture_condition', 'keep_test_genes', 'total_counts',
       'keep_effective_guides', 'donor_id', 'guide_sequence', 'guide_type',
       'keep_min_cells', 'perturbed_target_ensg', 'perturbed_target_symbol',
       'perturbed_target_biotype', 'perturbed_target_coord',
       'perturbed_target_chromosome', 'original_index',
       'perturbed_target_number', 'perturbed_target_chromosome_encoding'],
      dtype='object')

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        # treatment
        "treatment_label": None,
        "treatment_id": None,
        # replicates
        "technical_replicate": None,
        # "biological_replicate": None,
        # model system
        "model_system_label": "primary_cell",
        "model_system_id": None,
        "tissue": "lymphoid tissue",
        "cell_line_label": None,
        "cell_line_id": None,
        "cell_type_label": "CD4-positive, alpha-beta T cell",
        "disease_label": "healthy",
        "disease_id": None,

        "timepoint": None,
        "species": "Homo sapiens",
        # "sex_label": "male",
        "sex_id": None,
        # "developmental_stage_label": "adolescent",
        "developmental_stage_id": None,

        "study_title": "Genome-scale perturb-seq in primary human CD4+ T cells maps context-specific regulators of T cell programs and human immune traits",
        "study_uri": "https://doi.org/10.64898/2025.12.23.696273",
        "study_year": 2025,
        "first_author": "Ronghui Zhu",
        "last_author": "Alexander Marson",

        "experiment_title": "Perturb-seq of primary human CD4-positive T cells under resting and stimulated (for 8 and 48 hr) conditions",
        "experiment_summary": """
            Isolated human CD4-positive cells from four healthy donors were stimulated with ImmunoCult CD3/CD28/CD2 activator and sequentially transduced with dCas9-KRAB-Zim3 lentivirus (next morning after stimulation) and a Perturb-seq guide library (next afternoon after stimulation, MOI 0.2).
            The library consisted all genes expressed in human CD4+ T cells, all transcription factors annotated in the Lambert et al. (2018) and non-targeting controls totalling 12,748 genes.
            The gRNA sequences were selected from hCRISPRiv2 and Dolcetto libraries.
            On day 12, cells were split into three conditions:
            (1) Rest - cells were left for 8 hr without stimulation;
            (2) Stim8hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 8 hr;
            (3) Stim48hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 48 hr.
            Cells were harvested, fixed, stored using GEM-X Flex Sample Preparation v2 Kit and sequenced using GEM-X Flex Gene Expression Human n-plex kit converted into Ultima compatible libraries for sequencing on the Ultima Genomics UG100.
            """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_symbol'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-Zim3",

        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",

        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",

        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",

        "library_name": "custom",
        "library_uri": None,

        "library_format_id": None,
        "library_format_label": "pooled",

        "library_scope_id": None,
        "library_scope_label": "focused",

        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",

        "library_manufacturer": "Marson lab",
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",

        "readout_type_id": None,
        "readout_type_label": "transcriptomic",

        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",

        "method_name_id": None,
        "method_name_label": "Perturb-seq",

        "method_uri": None,

        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "GEM-X Flex Gene Expression Human n-plex kit",

        "sequencing_platform_id": None,
        "sequencing_platform_label": "Ultima Genomics UG100",

        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",

        "software_counts_id": None,
        "software_counts_label": "CellRanger",

        "software_analysis_id": None,
        "software_analysis_label": "scanpy",

        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",

        "license_label": "MIT License",
        "license_id": "SWO:9000074",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "Primary Human CD4+ T Cell Perturb-seq",
                "dataset_uri": "s3://genome-scale-tcell-perturb-seq/marson2025_data/GWCD4i.pseudobulk_merged.h5ad",
                "dataset_description": "Pseudobulk expression aggregated by guide, donor and culture condition.",
                "dataset_file_name": "GWCD4i.pseudobulk_merged.h5ad",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column treatment_label added to adata.obs
Column treatment_id added to adata.obs
Column technical_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_line_id added to adata.obs
Column cell_type_label added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_id added to adata.obs
Column study_title added to adata.obs
Column study_uri add

### Curate treatment information

In [ ]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['culture_condition'].map({
    'Rest':'untreated control',
    'Stim8hr': 'anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody',
    'Stim48hr': 'anti-CD3 antibody|anti-CD28 antibody|anti-CD2 antibody'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['culture_condition'].map({
    'Rest':'NCIT:C184729',
    'Stim8hr': 'EFO:0003317|EFO:0003304|NCIT:C184729',
    'Stim48hr': 'EFO:0003317|EFO:0003304|NCIT:C184729'
})

### Curate timepoints

In [ ]:
cur_data.adata.obs['timepoint'] = cur_data.adata.obs['culture_condition'].map({
    'Rest':'P12DT8H0M0S',
    'Stim8hr': 'P12DT8H0M0S',
    'Stim48hr': 'P14DT0H0M0S'
})

In [ ]:
cur_data.adata.obs

,log10_n_cells,developmental_stage_label,perturbed_gene_id,keep_for_DE,keep_total_counts,10xrun_id,perturbed_gene_name,perturbation_name,guide_id,sex_label,biological_replicate,n_cells,culture_condition,keep_test_genes,total_counts,keep_effective_guides,donor_id,guide_sequence,guide_type,keep_min_cells,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding,dataset_id,sample_id,perturbation_type_label,perturbation_type_id,data_modality,significant,significance_criteria,score_interpretation,treatment_label,treatment_id,technical_replicate,model_system_label,model_system_id,tissue,cell_line_label,cell_line_id,cell_type_label,disease_label,disease_id,timepoint,species,sex_id,developmental_stage_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,reference_genome_id,reference_genome_label,license_label,license_id,associated_datasets
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,1.414973,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-1,A1BG-1,female,D1_CE0008162,26.0,Stim48hr,True,326500.0,True,CE0008162,GGACGGCATCTCGGCCCGCC,targeting,True,ENSG00000121410,A1BG,protein_coding,chr19:58345178-58353492;-1,19,0,1,19,zhu_2025_pseudobulk,1,CRISPRi,None,Perturb-seq,None,None,None,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,None,primary_cell,None,lymphoid tissue,None,None,"CD4-positive, alpha-beta T cell",healthy,None,P14DT0H0M0S,Homo sapiens,None,None,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,\n Isolated human CD4-positive cell...,12731,278684,EFO:0022868,endogenous,None,dCas9-KRAB-Zim3,None,lentivirus transduction,None,lentivirus transduction,None,random locus integration,None,random locus integration,None,constitutive transgene expression,None,constitutive transgene expression,custom,None,None,pooled,None,focused,None,inhibition,Marson lab,2,2,25954,None,None,high-dimensional assay,None,transcriptomic,None,single-cell rna-seq,None,Perturb-seq,None,None,GEM-X Flex Gene Expression Human n-plex kit,None,Ultima Genomics UG100,None,barcode sequencing,None,CellRanger,None,scanpy,None,GRCh38,MIT License,SWO:9000074,"[{""dataset_accession"": ""Primary Human CD4+ T C..."
1,2.204120,adult,ENSG00000121410,True,True,CD4i_R2,A1BG,CD4i_R2_D1_Stim48hr_A1BG-2,A1BG-2,female,D1_CE0008162,160.0,Stim48hr,True,2313880.0,True,CE0008162,GGGTCCCTCGCAGCGCAGGA,targeting,True,ENSG00000121410,A1BG,protei

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
      input_column input_column_lower       name_lower     ontology_id
0  lymphoid tissue    lymphoid tissue  lymphoid tissue  UBERON:0001744
--------------------------------------------------


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 1 cell_type ontology terms from `cell_type_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
                      input_column               input_column_lower  \
0  CD4-positive, alpha-beta T cell  cd4-positive, alpha-beta t cell   

                        name_lower ontology_id  
0  cd4-positive, alpha-beta t cell  CL:0000624  
--------------------------------------------------
Overwriting column cell_type_label in adata.obs


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [ ]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

2026-02-09 16:10:32,971 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,zhu_2025_pseudobulk,1,Perturb-seq,<NA>,<NA>,CD4i_R2_D1_Stim48hr_A1BG-1,chr19:58345178-58353492;-1,19,19,1,ENSG00000121410,A1BG,protein_coding,GGACGGCATCTCGGCCCGCC,CRISPRi,<NA>,P14DT0H0M0S,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,<NA>,D1_CE0008162,primary_cell,<NA>,Homo sapiens,lymphoid tissue,UBERON:0001744,"CD4-positive, alpha-beta T cell",CL:0000624,<NA>,<NA>,female,<NA>,adult,<NA>,healthy,<NA>,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,Isolated human CD4-positive cells...,12731,278684,EFO:0022868,endogenous,<NA>,dCas9-KRAB-Zim3,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,custom,<NA>,<NA>,pooled,<NA>,focused,<NA>,inhibition,Marson lab,2,2,25954,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,GEM-X Flex Gene Expression Human n-plex kit,<NA>,Ultima Genomics UG100,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,scanpy,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""Primary Human CD4+ T C...",MIT License,SWO:9000074
1,zhu_2025_pseudobulk,2,Perturb-seq,<NA>,<NA>,CD4i_R2_D1_Stim48hr_A1BG-2,chr19:58345178-58353492;-1,19,19,1,ENSG00000121410,A1BG,protein_coding,GGGTCCCTCGCAGCGCAGGA,CRISPRi,<NA>,P14DT0H0M0S,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,<NA>,D1_CE0008162,primary_cell,<NA>,Homo sapiens,lymphoid tissue,UBERON:0001744,"CD4-positive, alpha-beta T cell",CL:0000624,<NA>,<NA>,female,<NA>,adult,<NA>,healthy,<NA>,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,Is

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ids",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['CUSTOM001_PuroR', 'ENSG00000148362', 'ENSG00000139656']; attempting to fetch latest IDs...
Fetched latest Ensembl IDs: {'ENSG00000148362': 'ENSG00000310560', 'ENSG00000139656': 'ENSG00000226519'}
--------------------------------------------------
Successfully mapped 18129 out of 18129 Ensembl IDs.
--------------------------------------------------


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
# replace gene symbols that are NOT ENSG with original provided gene symbol
cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_symbol'] = cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_name']

In [ ]:
cur_data.adata.var.loc[cur_data.adata.var['ensembl_gene_id'] == 'CUSTOM001_PuroR', 'ensembl_gene_id'] = None

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

2026-02-09 16:11:52,418 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,None,PuroR
1,ENSG00000000003,TSPAN6
2,ENSG00000000005,TNMD
3,ENSG00000000419,DPM1
4,ENSG00000000457,SCYL3
...,...,...
18124,ENSG00000291110,TRIM16L
18125,ENSG00000291122,CASTOR3P
18126,ENSG00000291135,FCGR1BP


# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

/content/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)
... storing 'dataset_id' as categorical
... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation_type_id' as categorical
... storing 'timepoint' as categorical
... storing 'treat

✅ Curated h5ad data saved to /content/PerturbationCatalogue/data_exploration/Perturbseq/curated/h5ad/zhu_2025_pseudobulk_curated.h5ad


In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to /content/PerturbationCatalogue/data_exploration/Perturbseq/curated/parquet/zhu_2025_pseudobulk_curated_metadata.parquet


# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='/content/PerturbationCatalogue/data_exploration/Perturbseq/curated/parquet/zhu_2025_pseudobulk_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file /content/PerturbationCatalogue/data_exploration/Perturbseq/curated/parquet/GWCD4i.pseudobulk_merged_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 278684 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [ ]:
!gcloud storage cp /content/PerturbationCatalogue/data_exploration/Perturbseq/curated/h5ad/zhu_2025_pseudobulk_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file:///content/PerturbationCatalogue/data_exploration/Perturbseq/curated/h5ad/zhu_2025_pseudobulk_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/zhu_2025_pseudobulk_curated.h5ad

Average throughput: 65.9MiB/s


In [ ]:
cur_data.adata.obs[cur_data.adata.obs['sample_id'] == '92281']

,dataset_id,sample_id,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,perturbed_target_number,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,guide_sequence,perturbation_type_label,perturbation_type_id,timepoint,treatment_label,treatment_id,technical_replicate,biological_replicate,model_system_label,model_system_id,species,tissue_label,tissue_id,cell_type_label,cell_type_id,cell_line_label,cell_line_id,sex_label,sex_id,developmental_stage_label,developmental_stage_id,disease_label,disease_id,study_title,study_uri,study_year,first_author,last_author,experiment_title,experiment_summary,number_of_perturbed_targets,number_of_perturbed_samples,library_generation_type_id,library_generation_type_label,library_generation_method_id,library_generation_method_label,enzyme_delivery_method_id,enzyme_delivery_method_label,library_delivery_method_id,library_delivery_method_label,enzyme_integration_state_id,enzyme_integration_state_label,library_integration_state_id,library_integration_state_label,enzyme_expression_control_id,enzyme_expression_control_label,library_expression_control_id,library_expression_control_label,library_name,library_uri,library_format_id,library_format_label,library_scope_id,library_scope_label,library_perturbation_type_id,library_perturbation_type_label,library_manufacturer,library_lentiviral_generation,library_grnas_per_target,library_total_grnas,library_total_variants,readout_dimensionality_id,readout_dimensionality_label,readout_type_id,readout_type_label,readout_technology_id,readout_technology_label,method_name_id,method_name_label,method_uri,sequencing_library_kit_id,sequencing_library_kit_label,sequencing_platform_id,sequencing_platform_label,sequencing_strategy_id,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
92280,zhu_2025_pseudobulk,92281,Perturb-seq,NaN,<NA>,CD4i_R1_D1_Stim8hr_SMIM11B-2,NaN,NaN,0,1,<NA>,SMIM11B,<NA>,CTTACTGCTCCCGGCAGCGG,CRISPRi,<NA>,P12DT8H0M0S,anti-CD3 antibody|anti-CD28 antibody|anti-CD2 ...,EFO:0003317|EFO:0003304|NCIT:C184729,<NA>,D1_CE0008162,primary_cell,<NA>,Homo sapiens,lymphoid tissue,UBERON:0001744,"CD4-positive, alpha-beta T cell",CL:0000624,<NA>,<NA>,female,<NA>,adult,<NA>,healthy,<NA>,Genome-scale perturb-seq in primary human CD4+...,https://doi.org/10.64898/2025.12.23.696273,2025,Ronghui Zhu,Alexander Marson,Perturb-seq of primary human CD4-positive T ce...,\n Isolated human CD4-positive cell...,12731,278684,EFO:0022868,endogenous,<NA>,dCas9-KRAB-Zim3,<NA>,lentivirus transduction,<NA>,lentivirus transduction,<NA>,random locus integration,<NA>,random locus integration,<NA>,constitutive transgene expression,<NA>,constitutive transgene expression,custom,<NA>,<NA>,pooled,<NA>,focused,<NA>,inhibition,Marson lab,2,2,25954,<NA>,<NA>,high-dimensional assay,<NA>,transcriptomic,<NA>,single-cell rna-seq,<NA>,Perturb-seq,<NA>,<NA>,GEM-X Flex Gene Expression Human n-plex kit,<NA>,Ultima Genomics UG100,<NA>,barcode sequencing,<NA>,CellRanger,<NA>,scanpy,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""Primary Human CD4+ T C...",MIT License,SWO:9000074


In [ ]:
cur_data.gene_ont[cur_data.gene_ont['synonym'] == 'SMIM11B']

,ensembl_gene_id,gene_symbol,chromosome_name,gene_coord,biotype,description,synonym_type,synonym
